In [1]:
import torch
import math

Scaled Dot-Product Attention

In [ ]:
class RawAttention(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, q, k, v, mask=None):
        """
        q shape: [Batch, Heads, Seq_len, Head_dim]
        k shape: [Batch, Heads, Seq_len, Head_dim]
        v shape: [Batch, Heads, Seq_len, Head_dim]
        mask shape: [Seq_len, Seq_len] (optional)
        """
        batch_size, num_heads, seq_len, head_dim = q.shape
        
        # Step 1: Calculate the dot product of Q and K transposed
        # Hint: You only want to transpose the last two dimensions of K. 
        dot_prod = q@k.transpose(-2,-1)
        # print(dot_prod.shape)
        # Look up k.transpose() or k.transpose(-2, -1)
        
        # Step 2: Scale the result by the square root of head_dim
        dot_prod_scaled = dot_prod/math.sqrt(head_dim)

        # Step 3: Apply the mask (if provided) by filling masked positions with -inf
        # Hint: scores.masked_fill_(mask == 0, float('-inf'))
        
        # Step 4: Apply softmax along the last dimension (dim=-1) to get probabilities
        dot_softmax = torch.softmax(dot_prod_scaled, dim=1)
        # Step 5: Multiply the resulting probabilities by V
        attn = dot_softmax @ v
        # return the final tensor

        return attn
        pass

In [3]:
batch, heads, seq, dim = 2, 4, 8, 16
q = torch.randn(batch, heads, seq, dim)
k = torch.randn(batch, heads, seq, dim)
v = torch.randn(batch, heads, seq, dim)

In [21]:
model = RawAttention()
output = model(q, k, v)

torch.Size([2, 4, 8, 8])


In [22]:
print("Output shape:", output.shape if output is not None else "Not implemented yet")

Output shape: torch.Size([2, 4, 8, 16])


The KV-Cache Engine

In [43]:
class KVCacheAttention(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, q, k, v, kv_cache=None):
        """
        In generation mode, sequence length of input q, k, v is ALWAYS 1.
        q shape: [Batch, Heads, 1, Head_dim]
        k shape: [Batch, Heads, 1, Head_dim]
        v shape: [Batch, Heads, 1, Head_dim]
        
        kv_cache is a tuple of (past_k, past_v).
        past_k shape: [Batch, Heads, Past_Seq_Len, Head_dim]
        """
        batch_size,head_num,seq_len,head_dim = q.shape

        if kv_cache is not None:
            past_k, past_v = kv_cache
            
            # Step 1: Update the cache
            # Use torch.cat() to glue past_k and the new k together along the sequence dimension (dim=2 or -2).
            # Do the same for v.
            k = torch.cat( [past_k,k], dim=-2 )
            v = torch.cat( [past_v,v], dim=-2 )
            pass
            
        # Store the newly updated k and v to return them for the next generation step
        new_kv_cache = (k, v)
        
        # Step 2: The Attention Math (Reuse your Milestone 1 logic!)
        # - Q @ K^T
        # - Scale by sqrt(head_dim)
        # - Softmax (dim=-1)
        # - @ V

        score = (q @ k.transpose(-2,-1))/ math.sqrt(head_dim)

        score_softmax = torch.softmax(score,dim=-1)

        final_score = score_softmax @ v

        # Note: We don't need a causal mask here! Because Q is only 1 token long, 
        # it is only looking at the past. It physically cannot look into the future.
        
        # return the final attention output AND the updated cache
        return final_score, new_kv_cache
        pass



In [44]:
# --- Test your code ---
batch, heads, dim = 2, 4, 16

# STEP 1: First token ("The")
q1 = torch.randn(batch, heads, 1, dim)
k1 = torch.randn(batch, heads, 1, dim)
v1 = torch.randn(batch, heads, 1, dim)

model = KVCacheAttention()
out1, cache = model(q1, k1, v1, kv_cache=None)

print("Output 1 shape:", out1.shape)        # Should be [2, 4, 1, 16]
print("Cached K shape:", cache[1].shape)    # Should be [2, 4, 1, 16]

Output 1 shape: torch.Size([2, 4, 1, 16])
Cached K shape: torch.Size([2, 4, 1, 16])


In [45]:
# STEP 2: Second token ("cat")
q2 = torch.randn(batch, heads, 1, dim)
k2 = torch.randn(batch, heads, 1, dim)
v2 = torch.randn(batch, heads, 1, dim)

# Pass in the cache from Step 1!
out2, cache = model(q2, k2, v2, kv_cache=cache)

print("Output 2 shape:", out2.shape)        # Should be [2, 4, 1, 16]
print("Cached K shape:", cache[0].shape)    # Should be [2, 4, 2, 16] <-- Notice sequence length grew to 2!

Output 2 shape: torch.Size([2, 4, 1, 16])
Cached K shape: torch.Size([2, 4, 2, 16])
